In [1]:
from DatascienceAllFunctions import DsFunctions as dsf
import pandas as pd
## Reading the data from CSV file
dataset=dsf.importDataset()

In [2]:
dataset.shape ## Tells the Number of Rows and Columns

(21000, 13)

In [3]:
dataset.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')

#### Step 1:Check for Null and duplicated records in the Dataset

#### 1.1 Find the information about the Dataset

In [4]:
dataset.info() ## Shows information about the table and dirty data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21000 entries, 0 to 20999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            19345 non-null  object 
 1   Gender             18846 non-null  object 
 2   Married            19309 non-null  object 
 3   Dependents         18832 non-null  object 
 4   Education          19278 non-null  object 
 5   Self_Employed      18151 non-null  object 
 6   ApplicantIncome    19403 non-null  float64
 7   CoapplicantIncome  19340 non-null  float64
 8   LoanAmount         19110 non-null  float64
 9   Loan_Amount_Term   19103 non-null  float64
 10  Credit_History     17791 non-null  float64
 11  Property_Area      19382 non-null  object 
 12  Loan_Status        21000 non-null  object 
dtypes: float64(5), object(8)
memory usage: 2.1+ MB


#### 1.2 Check for Duplicates

In [5]:
int(dataset.duplicated().sum())

9960

#### 1.3 Remove the duplicates

In [6]:
dataset = dataset.drop_duplicates()

In [7]:
int(dataset.duplicated().sum()) ## Rechecking if duplicate record exist or not

0

#### 1.4 Find NaN Values

In [8]:
dataset.isnull().sum()

Loan_ID              1350
Gender               1535
Married              1359
Dependents           1576
Education            1344
Self_Employed        1923
ApplicantIncome      1256
CoapplicantIncome    1318
LoanAmount           1410
Loan_Amount_Term     1374
Credit_History       2035
Property_Area        1273
Loan_Status             0
dtype: int64

####  2) Fix the dataset for Categorical data

#### 2.1. Split the data for quan and qual

In [9]:
### Note: Credit history is a boolean value and can be considered as a Categorical Data
### Moving it to Categorical data before Split

# Step 1: Create a clean categorical Series
credit_series = (pd.to_numeric(dataset['Credit_History'], errors='coerce').fillna(-1).astype(int)
    .map({1: 'Has_History', 0: 'No_History', -1: 'Unknown'}).astype('category'))

# Step 2: DROP the old column (important)
dataset = dataset.drop(columns=['Credit_History'])

# Step 3: Assign the new categorical column
dataset['Credit_History'] = credit_series

In [10]:
dataset['Credit_History'].dtype

CategoricalDtype(categories=['Has_History', 'No_History', 'Unknown'], ordered=False, categories_dtype=object)

In [11]:
## Droppig this column as it doesnt helps prediction. 
dataset.drop(columns=['Loan_Amount_Term'], inplace=True)

In [12]:
dataset

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Property_Area,Loan_Status,Credit_History
0,LP002604,Male,Yes,2,NaN,No,3166.000000,NaN,145.000000,Urban,N,Has_History
1,LP001787,Male,Yes,3+,NaN,No,3089.000000,NaN,100.000000,Rural,N,Has_History
2,LP002921,Male,Yes,3+,Not Graduate,No,5316.000000,187.000000,158.000000,Semiurban,N,No_History
3,LP002355,NaN,Yes,0,Graduate,No,3186.000000,3145.000000,150.000000,Semiurban,N,Unknown
4,LP002605,Male,No,0,Not Graduate,No,3271.000000,0.000000,90.000000,Rural,N,Has_History
...,...,...,...,...,...,...,...,...,...,...,...,...
20990,LP001857,Male,No,NaN,Not Graduate,Yes,1599.000000,2474.000000,2318.345334,Semiurban,N,Has_History
20991,LP001124,Female,NaN,3+,Not Graduate,No,2083.000000,14237.139301,28.000000,Urban,Y,Has_History
20997,LP002007,Male,Yes,2,Not Graduate,No,NaN,3333.000000,131.000000,Urban,N,Has_History
20998,LP002566,Female,No,0,Graduate,No,5530.000000,0.000000,135.000000,metro,N,Unknown


In [13]:
quan, qual = dsf.quanQual(dataset)

In [14]:
quan

['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

In [15]:
qual

['Loan_ID',
 'Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'Property_Area',
 'Loan_Status',
 'Credit_History']

#### 2.2 Check for Null values

In [16]:
dataset.isnull().sum()

Loan_ID              1350
Gender               1535
Married              1359
Dependents           1576
Education            1344
Self_Employed        1923
ApplicantIncome      1256
CoapplicantIncome    1318
LoanAmount           1410
Property_Area        1273
Loan_Status             0
Credit_History          0
dtype: int64

####     2.3.Fill Null Values with Mode for Categorical data

In [17]:
for col in qual:
    dataset.loc[:, col] = dataset[col].fillna(dataset[col].mode()[0])

#### 2.4 Cleaning up Gender Column

In [18]:
dataset['Gender'].value_counts()

Gender
Male       8452
Female     1712
Unknown     876
Name: count, dtype: int64

In [19]:
## Fill unknown with Mode
dataset.loc[:, 'Gender'] = dataset['Gender'].fillna(dataset['Gender'].mode()[0])

#### 2.5 Cleanup Credit History Data

In [20]:
dataset['Credit_History'].value_counts()

Credit_History
Has_History    7451
Unknown        2035
No_History     1554
Name: count, dtype: int64

In [21]:
## Fill unknown with Mode

mode_value = dataset['Credit_History'].mode()[0]
dataset.loc[:,'Credit_History'] = dataset['Credit_History'].replace('Unknown',mode_value)

C:\Users\xsanthkum\AppData\Local\Temp\ipykernel_31972\2280655078.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  dataset.loc[:,'Credit_History'] = dataset['Credit_History'].replace('Unknown',mode_value)
C:\Users\xsanthkum\AppData\Local\Temp\ipykernel_31972\2280655078.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Has_History', 'Has_History', 'No_History', 'Has_History', 'Has_History', ..., 'Has_History', 'Has_History', 'Has_History', 'Has_History', 'Has_History']
Length: 11040
Categories (2, object): ['Has_History', 'No_History']' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  dataset.loc[:,'Credit_History'] = dataset['Credit_History'].replace('Unkn

In [22]:
dataset['Credit_History'].value_counts()

Credit_History
Has_History    9486
No_History     1554
Name: count, dtype: int64

#### 3. Fix the data for Numerical data using Median

In [23]:

for col in quan:
    dataset.loc[:, col] = dataset[col].fillna(dataset[col].median())

In [24]:
### Reverify the dataset
dataset.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Property_Area        0
Loan_Status          0
Credit_History       0
dtype: int64

#### 4. Write cleaned Dataset to CSV

In [25]:
dataset.to_csv("Loan_Prediction_Cleaned.csv", index=False)